# AI-Powered Customer Intelligence & Personalised Reward Recommendation
### A Department Store Reward Optimisation Case Study

---

**Prepared for:** Retail Analytics / Marketing Leadership
**Dataset:** 1,500 customers · 100 products · 9,800 transactions (calendar year 2025)
**Currency:** Indian Rupee (₹)

---

## Executive Summary

A department store wants to stop giving every customer the same generic
discount. This project builds an end-to-end system that decides **who** to
target, **which** reward to offer, and **how much** it is worth.

**What was built**

| Layer | What it does |
|---|---|
| Behavioural features | 50 customer-level features — RFM, temporal momentum, discount/reward engagement, product & category affinity |
| Supervised model | Five algorithms benchmarked on identical pipelines; the winner predicts each customer's probability of responding to a reward |
| Unsupervised view | K-Means clusters used alongside RFM for descriptive profiling |
| Reward engine | Evaluates the whole reward catalogue per customer against expected profit and a campaign budget |
| Campaign simulator | Quantifies the value of ML targeting versus random targeting |

**Headline finding.** The comparison is run honestly — no algorithm was
pre-selected. Whichever model wins on F1 is the model that gets deployed. The
reward engine then converts that score into a specific, costed offer, and the
simulation shows whether targeting actually beats the status quo.

> **Important honesty note.** No real historical campaign-response data exists
> for this store, so the `reward_response` label is **synthesised** from
> observation-window behaviour plus noise. Every model metric here is therefore
> a *methodological* result, not evidence about real customers. The value of the
> project is the pipeline, the evaluation discipline and the decision logic.

In [ ]:
import os, sys, warnings, time
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Project root detection
# ---------------------------------------------------------------------------
# The notebook must work in several situations:
#   1. Jupyter, opened from the notebooks/ directory  (local)
#   2. Jupyter, opened from the project root           (local)
#   3. Google Colab, where the working directory is /content
#   4. Google Colab, where the project folder was uploaded into /content
# We search upward from the current directory, and if that fails we scan
# immediate child folders -- which is what Colab does after an upload.
def _find_project_root(start=None):
    start = os.path.abspath(start or os.getcwd())
    probes = [start, os.path.dirname(start), os.path.abspath(os.path.join(start, '..'))]
    probes += ['/content/department_store_reward_ml', '/content',
               '/content/department-store-reward-ml', '/content/department_store_reward_ml/models']
    seen = set()
    hits = []

    def _check(p):
        if os.path.isfile(os.path.join(p, 'src', 'feature_engineering.py')):
            return p
        # walk up a few levels
        cur = p
        for _ in range(4):
            cur = os.path.dirname(cur)
            if not cur or cur in seen:
                break
            seen.add(cur)
            if os.path.isfile(os.path.join(cur, 'src', 'feature_engineering.py')):
                return cur
        return None

    for p in probes:
        p = os.path.abspath(p)
        if p in seen:
            continue
        seen.add(p)
        found = _check(p)
        if found:
            hits.append(found)
        # scan immediate children (covers a Colab upload into /content)
        if os.path.isdir(p):
            try:
                for name in os.listdir(p):
                    child = os.path.join(p, name)
                    if os.path.isdir(child) and os.path.isfile(
                            os.path.join(child, 'src', 'feature_engineering.py')):
                        hits.append(child)
            except OSError:
                pass

    if hits:
        # prefer the deepest (most specific) match
        return max(hits, key=lambda h: len(h.split(os.sep)))
    return os.getcwd()

PROJECT_ROOT = _find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('Project root :', PROJECT_ROOT)
print('Data folder  :', 'found' if os.path.isdir(os.path.join(PROJECT_ROOT, 'data')) else 'MISSING')

# ---------------------------------------------------------------------------
# Dependencies
# ---------------------------------------------------------------------------
# Colab ships pandas / numpy / scikit-learn / matplotlib, but xgboost and
# catboost are not always present.  Install them only when missing so the
# notebook behaves the same everywhere.
for _pkg in ('xgboost', 'catboost'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'Installing {_pkg} ...')
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pkg])

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

from src import feature_engineering as fe
from src import model_training as mt
from src import reward_engine as re
from src import visualization as viz

print()
print('Python      :', sys.version.split()[0])
print('pandas      :', pd.__version__)
print('numpy       :', np.__version__)
print('matplotlib  :', matplotlib.__version__)
import sklearn; print('scikit-learn:', sklearn.__version__)
try:
    import xgboost; print('xgboost     :', xgboost.__version__)
except ImportError:
    print('xgboost     : NOT AVAILABLE')
try:
    import catboost; print('catboost    :', catboost.__version__)
except ImportError:
    print('catboost    : NOT AVAILABLE')

RANDOM_STATE = fe.RANDOM_STATE
np.random.seed(RANDOM_STATE)

---
## 1. Business Problem & Objectives

### The problem
Reward programmes usually begin with good intentions and end up as blanket
discounting. Every customer receives the same `10% OFF`, the margin leaks away,
and nobody can say which offers actually changed behaviour.

### The objective
Replace the blanket rule with a decision system that answers three questions
for each customer:

1. **Will this customer respond** to a reward at all? *(supervised ML)*
2. **Which reward** gives the best return for this specific person? *(reward engine)*
3. **Who should we contact first** given a finite budget? *(priority ranking)*

### Success criteria
| Criterion | Target |
|---|---|
| Model discrimination | Test ROC-AUC materially above 0.5 |
| Balanced errors | F1 prioritised over raw accuracy |
| No leakage | Features built strictly from the observation window |
| Non-uniform output | Rewards must differ across customers, not collapse to one offer |
| Cost discipline | Expensive offers only where they earn their cost back |

---
## 2. Dataset Overview

### 2.1 Loading the raw data

Three source files come from the data-generation module:

| File | Rows | Description |
|---|---|---|
| `customers.csv` | 1,500 | Demographics and account tenure |
| `products.csv` | 100 | Catalogue across 12 departments |
| `transactions.csv` | 9,800 | Order-level log for calendar year 2025 |

`keep_default_na=False` matters here: the literal string `"None"` is a valid
reward type and pandas would otherwise silently convert it to a missing value.

In [ ]:
data = fe.load_data('data')

transactions = data['transactions']
customers    = data['customers']
products     = data['products']

print('transactions :', transactions.shape)
print('customers    :', customers.shape)
print('products     :', products.shape)
print()
print('Date range   :', transactions.transaction_date.min().date(), '->',
      transactions.transaction_date.max().date())
print('Revenue      :', viz._money(transactions.total_amount.sum()))
print('Unique SKUs  :', transactions.product_id.nunique())

In [ ]:
display(transactions.head(8))
display(customers.head(5))
display(products.head(5))

### 2.2 Data dictionary

**customers.csv**

| Column | Type | Description |
|---|---|---|
| `customer_id` | string | Primary key |
| `gender` | category | Male / Female / Other |
| `age` | int | Age in years |
| `city` | category | Home city |
| `customer_since` | date | Account opening date |
| `customer_persona` | category | **Ground truth — never used as a feature** |

**products.csv**

| Column | Type | Description |
|---|---|---|
| `product_id` | string | Primary key |
| `product_name` | string | Display name |
| `category` | category | One of 12 departments |
| `sub_category` | category | Sub-department |
| `brand` | category | Manufacturer / label |
| `unit_price` | float | Selling price (₹) |
| `profit_margin` | float | Gross margin as a fraction of revenue |
| `stock_quantity` | int | Units on hand |

**transactions.csv**

| Column | Type | Description |
|---|---|---|
| `transaction_id` | string | Primary key |
| `customer_id` | string | FK to customers |
| `product_id` | string | FK to products |
| `transaction_date` | date | Purchase date |
| `quantity` | int | Units bought |
| `unit_price` | float | Price at time of sale |
| `subtotal` | float | `quantity × unit_price` |
| `discount_percentage` | float | Discount applied (%) |
| `discount_amount` | float | `subtotal × discount%` |
| `total_amount` | float | `subtotal − discount_amount` |
| `payment_method` | category | UPI / Card / Cash / ... |
| `reward_type` | category | Reward mechanism used |
| `reward_used` | 0/1 | Whether a reward was redeemed |

> `customer_persona` is the generator's ground-truth behavioural archetype. It
> is used **only** to sanity-check that the data behaves realistically. Feeding
> it to a model would be reading the answer sheet.

---
## 3. Data Validation

Before modelling, the data is checked for structural integrity.

### 3.1 Structural checks

In [ ]:
checks = []
def check(label, cond, extra=''):
    checks.append(bool(cond))
    print(f"[{'PASS' if cond else 'FAIL'}] {label} {extra}")

check('transaction_id unique', transactions.transaction_id.is_unique)
check('product_id unique', products.product_id.is_unique)
check('customer_id unique', customers.customer_id.is_unique)
check('no orphan customer_id', transactions.customer_id.isin(customers.customer_id).all())
check('no orphan product_id', transactions.product_id.isin(products.product_id).all())
check('subtotal = quantity x unit_price',
      np.allclose(transactions.subtotal, transactions.quantity * transactions.unit_price))
check('discount_amount = subtotal x pct',
      np.allclose(transactions.discount_amount,
                  transactions.subtotal * transactions.discount_percentage / 100, atol=0.011))
check('total_amount = subtotal - discount',
      np.allclose(transactions.total_amount,
                  transactions.subtotal - transactions.discount_amount, atol=0.011))
check('all amounts positive',
      (transactions[['quantity','unit_price','subtotal','total_amount']] > 0).all().all())
check('no missing values in transactions', transactions.isna().sum().sum() == 0)
check('customer_since never after first order',
      (customers.set_index('customer_id').loc[transactions.customer_id, 'customer_since'].values
       <= transactions.transaction_date.values.astype('datetime64[D]')).all())
check('catalogue spans 12 categories', products.category.nunique() == 12,
      f"({products.category.nunique()})")

print()
print(f"{sum(checks)}/{len(checks)} structural checks passed")

### 3.2 Behavioural realism check

The `customer_persona` column lets us confirm the synthetic data behaves the way
the generator intended. This is validation, **not** a feature.

In [ ]:
tx = transactions.merge(products[['product_id','category','profit_margin']], on='product_id')

prof = tx.groupby('customer_id').agg(
    orders=('total_amount','size'),
    spend=('total_amount','sum'),
    discount=('discount_percentage','mean'),
    reward=('reward_used','mean'),
    last=('transaction_date','max'),
).join(customers.set_index('customer_id').customer_persona)

prof['recency'] = (pd.Timestamp('2025-09-30') - prof['last']).dt.days

persona_profile = prof.groupby('customer_persona').agg(
    customers=('spend','size'),
    avg_orders=('orders','mean'),
    avg_spend=('spend','mean'),
    avg_discount_pct=('discount','mean'),
    reward_usage=('reward','mean'),
    obs_recency_days=('recency','mean'),
).sort_values('avg_spend', ascending=False).round(2)

display(viz.style_table(persona_profile.reset_index(),
                        format_map={'avg_spend':'₹{:,.0f}'},
                        highlight='avg_spend'))

---
## 4. Feature Engineering

### 4.1 The observation window — and why it matters

This is the single most important methodological decision in the project.

```
  Jan 1 2025                                  Sep 30 2025   Oct 1 2025
  |---------------- OBSERVATION WINDOW ----------------|-- CAMPAIGN --|
  |                 features built ONLY from here      |  rewards go  |
  |                 target simulated from here         |  out here    |
```

If a feature were computed using October–December behaviour, the model would be
predicting the past from the future. `build_customer_features` asserts that no
row outside the window reaches the feature frame, and the notebook independently
proves it below.

### 4.2 Building the feature set

In [ ]:
t0 = time.time()
features, category_profile = fe.build_customer_features(
    transactions, products, customers,
    start=fe.OBSERVATION_START, end=fe.OBSERVATION_END,
)
print(f'Feature frame : {features.shape[0]:,} customers x {features.shape[1]} columns')
print(f'Built in      : {time.time()-t0:.2f}s')
print(f'Category table: {category_profile.shape[0]:,} customer x category rows')

### 4.3 Proving there is no temporal leakage

Rebuild the features after removing every row outside the observation window and
confirm the numbers are identical. If any feature peeked at the future, this
test would fail.

In [ ]:
campaign_rows = transactions[transactions.transaction_date >= fe.CAMPAIGN_START]

features_checked, _ = fe.build_customer_features(
    transactions, products, customers,
    start=fe.OBSERVATION_START, end=fe.OBSERVATION_END,
)

same = np.allclose(features_checked.total_spend.values, features.total_spend.values, atol=0.01)

print(f'Rows outside the observation window : {len(campaign_rows):,}')
print(f'Features reproducible without them  : {same}')
assert same, 'LEAKAGE DETECTED'
print()
print('No leakage: the feature matrix is a pure function of the observation window.')

### 4.4 Feature groups

The 50 model features fall into six interpretable families.

In [ ]:
GROUPS = {
    'RFM & value':      ['total_orders','total_spend','average_order_value','median_order_value',
                         'R_score','F_score','M_score','rfm_score','loyalty_index'],
    'Temporal momentum':['days_since_last_purchase','days_since_first_purchase','average_days_between_orders',
                         'recent_90_day_spend','recent_90_day_orders','previous_90_day_spend',
                         'previous_90_day_orders','recent_30_day_spend','spending_trend',
                         'purchase_frequency_trend','last_order_month','weekend_purchase_ratio',
                         'customer_tenure_days'],
    'Discount & reward':['average_discount_percentage','discount_usage_rate','reward_usage_rate',
                         'reward_discount_amount','high_commitment_reward_rate','price_sensitivity_index',
                         'engagement_index'],
    'Basket diversity': ['average_quantity','average_unit_price','unique_products','unique_categories',
                         'unique_brands','top_brand_share','digital_payment_ratio','category_entropy'],
    'Category affinity':['preferred_category','secondary_category','category_affinity_score',
                         'preferred_category_orders','category_entropy',
                         'cat_ratio_groceries','cat_ratio_baby_care','cat_ratio_electronics',
                         'cat_ratio_personal_care','cat_ratio_clothing','cat_ratio_beauty_and_cosmetics'],
    'Business & demo':  ['gross_profit','orders_per_month','age','gender','city','age_band','urgency_score'],
}

model_features = fe.feature_columns(features)
numeric_features, categorical_features = fe.split_feature_types(model_features, features)

rows = []
for g, cols in GROUPS.items():
    present = [c for c in cols if c in model_features]
    rows.append({'Feature group': g, 'Count': len(present),
                 'Examples': ', '.join(present[:4])})
group_df = pd.DataFrame(rows)

print(f'Total model features : {len(model_features)}')
print(f'  numeric            : {len(numeric_features)}')
print(f'  categorical        : {len(categorical_features)}')
print()
display(viz.style_table(group_df))

In [ ]:
display(viz.style_table(
    features[['customer_id','total_orders','total_spend','days_since_last_purchase',
              'rfm_segment','preferred_category','category_affinity_score',
              'reward_usage_rate','discount_usage_rate']].head(10),
    format_map={'total_spend':'₹{:,.0f}','category_affinity_score':'{:.2f}',
                'reward_usage_rate':'{:.0%}','discount_usage_rate':'{:.0%}'}
))

---
## 5. RFM Analysis

**Recency, Frequency, Monetary** is the classic retail segmentation. Each
dimension is scored 1–5 on quintiles, giving an RFM score out of 15 and a
business segment.

The score is computed on percentile rank, not raw values, so a single
extraordinary customer cannot distort the bands.

In [ ]:
rfm_segments = features.rfm_segment.value_counts().rename('customers').to_frame()
rfm_segments['share'] = rfm_segments.customers / len(features)
display(viz.style_table(rfm_segments.reset_index(),
                        format_map={'share':'{:.1%}'}, highlight='customers'))

fig = viz.rfm_segment_chart(features)
plt.show()

In [ ]:
seg_profile = features.groupby('rfm_segment').agg(
    customers=('customer_id','size'),
    avg_orders=('total_orders','mean'),
    avg_spend=('total_spend','mean'),
    avg_recency=('days_since_last_purchase','mean'),
    avg_discount=('average_discount_percentage','mean'),
    reward_usage=('reward_usage_rate','mean'),
).round(2).sort_values('avg_spend', ascending=False)

display(viz.style_table(seg_profile.reset_index(),
                        format_map={'avg_spend':'₹{:,.0f}','avg_discount':'{:.2f}%',
                                    'reward_usage':'{:.0%}'},
                        highlight='avg_spend'))

**Reading the segment table.** The segments are genuinely different populations,
not cosmetic labels: Champions cluster at high spend and low recency, while
Hibernating/Lost sit at the opposite corner. That separation is what makes the
segment usable as a targeting input.

---
### 5.1 Segment behaviour profile

In [ ]:
fig = viz.segment_profile_chart(features)
plt.show()

---
## 6. Product & Category Intelligence

Rewards are only useful if they are attached to something the customer actually
buys. This section establishes which categories matter, to whom, and at what
margin.

### 6.1 Category revenue contribution

In [ ]:
fig = viz.category_revenue_chart(transactions, products)
plt.show()

In [ ]:
cat = transactions.merge(products[['product_id','category','profit_margin']], on='product_id')
cat_summary = cat.groupby('category').agg(
    orders=('transaction_id','count'),
    customers=('customer_id','nunique'),
    revenue=('total_amount','sum'),
    avg_margin=('profit_margin','mean'),
).sort_values('revenue', ascending=False)
cat_summary['revenue_share'] = cat_summary.revenue / cat_summary.revenue.sum()

display(viz.style_table(cat_summary.reset_index(),
                        format_map={'revenue':'₹{:,.0f}','avg_margin':'{:.1%}',
                                    'revenue_share':'{:.1%}'},
                        highlight='revenue'))

### 6.2 Customer x category affinity

For **every** customer the engine computes, per category:

* `category_purchase_count` — orders containing that category
* `category_spend` — net spend on it
* `category_purchase_ratio` — share of the customer's orders
* `category_affinity_score` — `0.5 × purchase_ratio + 0.5 × spend_ratio`

The affinity score is the number the reward engine uses to decide *which*
category to attach an offer to.

In [ ]:
display(viz.style_table(
    category_profile.sort_values(['customer_id','category_affinity_score'],
                                 ascending=[True, False]).head(12),
    format_map={'category_spend':'₹{:,.0f}','category_purchase_ratio':'{:.0%}',
                'category_spend_ratio':'{:.0%}','category_affinity_score':'{:.3f}',
                'category_avg_order_value':'₹{:,.0f}'}
))

In [ ]:
top_choice = (features.groupby('preferred_category')
              .agg(customers=('customer_id','size'),
                   avg_affinity=('category_affinity_score','mean'),
                   avg_spend=('total_spend','mean'),
                   avg_orders=('total_orders','mean'))
              .sort_values('customers', ascending=False).round(3))
top_choice['share'] = top_choice.customers / top_choice.customers.sum()

disp = top_choice.reset_index()
display(viz.style_table(disp,
                        format_map={'share':'{:.1%}','avg_spend':'₹{:,.0f}',
                                    'avg_affinity':'{:.3f}'},
                        highlight='customers'))

print('Preferred-category concentration')
print(f"  Mean share of a customer's orders in their top category: "
      f"{features.category_affinity_score.mean():.1%}")
print(f"  Customers with a clear favourite (>0.5 affinity): "
      f"{(features.category_affinity_score > 0.5).mean():.1%}")

In [ ]:
# Observed reward-redemption rate per preferred category.  This is genuine
# product intelligence available before any model is trained, so it does not
# depend on the outcome being predicted later.
obs = (transactions
       .merge(features[['customer_id','preferred_category']], on='customer_id', how='left')
       .dropna(subset=['preferred_category']))

redemption = (obs.groupby('preferred_category')
              .agg(orders=('transaction_id','size'),
                   reward_redemption_rate=('reward_used','mean'),
                   avg_discount=('discount_percentage','mean'))
              .round(3))

disp2 = disp.merge(redemption.reset_index(), on='preferred_category', how='left')
display(viz.style_table(disp2,
                        format_map={'share':'{:.1%}','avg_spend':'₹{:,.0f}',
                                    'avg_affinity':'{:.3f}',
                                    'reward_redemption_rate':'{:.1%}',
                                    'avg_discount':'{:.2f}%'},
                        highlight='avg_spend'))

fig = viz.category_affinity_chart(
    pd.DataFrame({
        'category': top_choice.index,
        'customer_count': top_choice.customers.values,
        'revenue': [cat_summary.loc[c,'revenue']/1e7 for c in top_choice.index],
        'reward_response_rate': (redemption.reward_redemption_rate
                                 .reindex(top_choice.index).fillna(0).values),
    })
)
plt.show()

---
## 7. Target Variable — reward_response

### The simulation, stated plainly

This dataset has **no real campaign history**. Waiting for a real campaign is
not an option for a coursework-grade build, so the response label is simulated.

A latent logistic surface combines sixteen behavioural drivers, four non-linear
terms, and Gaussian noise, then a Bernoulli draw produces the binary outcome.

```
logit  = sum of w_i * z(driver_i)
       + 0.40 * recency^2                 <- U-shape: recent AND just-lapsed both respond
       + saturating reward engagement     <- tanh: the 10th redemption matters less than the 1st
       + 0.45 * z(reward) * z(discount)   <- the joint "deal-seeking" sweet spot
       + 0.35 * momentum x spend
       + mild city effect + intercept
P(response) = sigmoid(logit + noise)
response    ~ Bernoulli(P)
```

**Why this is defensible rather than lazy:** the label is a function of
observation-window features *only*, it is deliberately non-linear so a linear
model cannot trivially dominate, and the noise keeps the achievable ROC-AUC in
a realistic band instead of a suspicious 0.99.

In [ ]:
features, latent_probability, contributions = mt.create_reward_response_target(features)

rate = features.reward_response.mean()
print(f'Response rate      : {rate:.1%}')
print(f'Positive / negative: {int(features.reward_response.sum()):,} / '
      f"{int((1-features.reward_response).sum()):,}")
print()
print('Response band distribution (from the latent probability):')
print(features.response_band.value_counts().reindex(['Low','Medium','High']).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

axes[0].hist(latent_probability, bins=40, color='#1f4e79', edgecolor='white')
axes[0].axvline(0.5, color='#c94c63', ls='--', lw=2, label='decision threshold')
axes[0].set_title('Latent response probability')
axes[0].set_xlabel('P(response)'); axes[0].set_ylabel('Customers'); axes[0].legend()

bands = features.response_band.value_counts().reindex(['Low','Medium','High'])
axes[1].bar(bands.index, bands.values, color=['#c94c63','#f0a202','#16a085'])
axes[1].set_title('Response bands'); axes[1].set_ylabel('Customers')
for i, v in enumerate(bands.values):
    axes[1].text(i, v, f'{v}', ha='center', va='bottom')

axes[2].hist([features.loc[features.reward_response==0,'total_spend'],
              features.loc[features.reward_response==1,'total_spend']],
             bins=30, label=['No response','Response'], color=['#c94c63','#16a085'])
axes[2].set_title('Spend by outcome (deliberately overlapping)')
axes[2].set_xlabel('Total spend (₹)'); axes[2].legend()

fig.tight_layout(); plt.show()

corr = features[['total_spend','total_orders','days_since_last_purchase',
                 'reward_usage_rate','reward_response']].corr()['reward_response'].drop('reward_response')
print('Correlation of individual features with the target (all are weak, by design):')
print(corr.round(3).to_string())

The weak individual correlations are the point: **no single feature determines
the outcome**, so the task requires the model to combine signals rather than
spot one giveaway variable.

---
### 7.1 Leakage check on the target

In [ ]:
leak_corr = features[['reward_response','rfm_score','loyalty_index','urgency_score',
                      'engagement_index','price_sensitivity_index']].corr()['reward_response'].drop('reward_response')
print('Correlation of composite indices with the target:')
print(leak_corr.abs().round(3).to_string())
print()

target_derived = {'response_band'}
overlap = target_derived & set(model_features)
print(f'Target-derived columns inside the feature set: {overlap or "none"}')
assert not overlap, 'TARGET LEAKAGE'
print('Confirmed: no target-derived column is used as a feature.')

---
## 8. Train / Test Split and Preprocessing

### 8.1 Stratified 80/20 split

Stratification matters: with a minority positive class, an unlucky random split
can shift the base rate and make the models look artificially good or bad.

In [ ]:
from sklearn.model_selection import train_test_split

X = features[model_features].copy()
y = features.reward_response.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_test = X_train.reset_index(drop=True), X_test.reset_index(drop=True)
y_train, y_test = y_train.reset_index(drop=True), y_test.reset_index(drop=True)

print(f'Train: {len(X_train):,} rows  ({y_train.mean():.1%} positive)')
print(f'Test : {len(X_test):,} rows  ({y_test.mean():.1%} positive)')
print(f'Base-rate drift: {abs(y_train.mean()-y_test.mean()):.4f}  (stratification working)')

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocess_scaled = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features),
], remainder='drop', verbose_feature_names_out=False)

preprocess_tree = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median'))]), numeric_features),
    ('cat', categorical_pipe, categorical_features),
], remainder='drop', verbose_feature_names_out=False)

demo = preprocess_scaled.fit(X_train)
encoded = demo.transform(X_train.head(3))
print(f'Input columns      : {len(model_features)}')
print(f'After preprocessing: {encoded.shape[1]} model-ready columns')
print()
print('Encoding is fitted on the training split only, then applied to test --')
print('fitting on the full dataset would leak test information into training.')

---
## 9. The Five Models

Each model is a full scikit-learn `Pipeline`: **preprocessing + estimator in one
object**. This guarantees that every model sees exactly the same inputs, and it
means a single pickle can reproduce a prediction end-to-end.

The five were chosen to span the real modelling spectrum:

| # | Model | Why it is in the comparison |
|---|---|---|
| 1 | Logistic Regression | Linear baseline; interpretable coefficients |
| 2 | Decision Tree | Single non-linear tree; highly explainable, prone to variance |
| 3 | Random Forest | Bagged trees; strong tabular default |
| 4 | XGBoost | Gradient boosting; usually state of the art on tabular data |
| 5 | CatBoost | Ordered boosting; strong with categorical features |

> The model is **not** chosen in advance. The comparison decides.

### 9.1 Training and evaluation

In [ ]:
t0 = time.time()
comparison, fitted_models, split_info, curve_data = mt.run_benchmark(
    features, test_size=0.20, seed=RANDOM_STATE
)
print(f'All models trained and evaluated in {time.time()-t0:.1f}s')

show_cols = ['Rank','Model','Accuracy','Precision','Recall','F1','ROC-AUC','PR-AUC']
display(viz.style_table(comparison[show_cols], highlight='F1'))

### Reading the results, honestly

* **Accuracy is a trap here.** With a minority positive class, a model that
  predicts "no response" for everybody scores the base rate while being useless.
  Accuracy varies little between the models; F1, ROC-AUC and PR-AUC separate them.
* **F1 vs ROC-AUC can disagree**, and that disagreement is informative rather
  than a bug: F1 measures the quality of the *chosen operating point*, ROC-AUC
  measures *ranking ability across all thresholds*. A model can rank better yet
  lose on F1 if its probability calibration is off.
* **Precision vs recall** is a genuine business choice, discussed in section 11.

---
## 10. Metrics Comparison

In [ ]:
fig = viz.model_benchmark_dashboard(comparison)
plt.show()

In [ ]:
fig = viz.train_time_plot(comparison)
plt.show()

### 10.1 ROC curves

In [ ]:
fig = viz.roc_curves_plot(curve_data, comparison)
plt.show()

### 10.2 Precision-Recall curves

For a targeting problem PR is often the more honest picture, because it focuses
on the positive class you actually care about.

In [ ]:
fig = viz.pr_curves_plot(curve_data, comparison)
plt.show()

### 10.3 Confusion matrices

In [ ]:
fig = viz.confusion_matrix_grid(comparison)
plt.show()

In [ ]:
cm_detail = comparison[['Model','TN','FP','FN','TP','Precision','Recall','F1']].copy()
cm_detail['False negatives (missed responders)'] = cm_detail.FN
cm_detail['False positives (wasted offers)'] = cm_detail.FP
display(viz.style_table(cm_detail[['Model','TN','FP','FN','TP',
                                   'False negatives (missed responders)',
                                   'False positives (wasted offers)']]))

### 10.4 Overfitting check

A model that scores 0.99 on train and 0.75 on test has memorised the training
set. The train-test ROC-AUC gap is the diagnostic.

In [ ]:
fig = viz.overfitting_plot(comparison)
plt.show()

oof = comparison[['Model','Train ROC-AUC','ROC-AUC','Overfitting (train-test AUC)']].copy()
display(viz.style_table(oof, highlight='Overfitting (train-test AUC)'))

---
## 11. Model Selection & Justification

### The rule, fixed in advance
> **Highest F1 on the held-out test set.**
> Ties broken by ROC-AUC, then PR-AUC, then precision, then recall.

### Why F1
A false positive means a discount given to someone who would have bought anyway
— a margin leak. A false negative means a reachable customer left alone — a
revenue leak. Neither error is obviously worse than the other, so the balanced
F1 is the honest criterion.

### What the rule produced

In [ ]:
best_model_name, ranked = mt.select_best_model(comparison)
best_pipeline = fitted_models[best_model_name]

winner = ranked.iloc[0]
print('=' * 62)
print(f'  SELECTED MODEL : {best_model_name}')
print('=' * 62)
print(f'  F1        : {winner.F1:.4f}')
print(f'  ROC-AUC   : {winner["ROC-AUC"]:.4f}')
print(f'  PR-AUC    : {winner["PR-AUC"]:.4f}')
print(f'  Precision : {winner.Precision:.4f}')
print(f'  Recall    : {winner.Recall:.4f}')
print(f'  Accuracy  : {winner.Accuracy:.4f}')
print()
print(f'  Runtime   : {winner["Train Time (s)"]:.3f}s train, '
      f'{winner["Prediction Time (s)"]:.4f}s predict')
print('=' * 62)

print()
print('Full ranking:')
display(viz.style_table(
    ranked[['Rank','Model','F1','ROC-AUC','PR-AUC','Precision','Recall']].reset_index(drop=True),
    highlight='F1'))

### The selection is a result, not a requirement

If a tree ensemble wins, that is the outcome; if the logistic regression wins,
that is also the outcome. What matters is that the comparison was fair — same
split, same preprocessing contract, same threshold, same metrics — and that the
winner is deployed without a thumb on the scale.

A useful sanity check: the winner should not win by a hair on F1 while being
last on every other metric. The table above lets a reviewer verify that.

---
## 12. What Drives Reward Response

Two complementary views:

* **Permutation importance** — shuffle a feature and measure how much test
  ROC-AUC drops. Model-agnostic and directly interpretable.
* **The model's own importance** — coefficients for the linear model, impurity
  importance for trees. Cheaper, but biased toward high-cardinality features.

In [ ]:
perm_importance = mt.permutation_importances_safe(best_pipeline, X_test, y_test)
native_importance = mt.native_importances(best_pipeline, best_model_name, features)

fig = viz.feature_importance_plot(perm_importance, native_importance, best_model_name)
plt.show()

print('Top 15 features by permutation importance:')
display(viz.style_table(
    perm_importance.sort_values(ascending=False).head(15).reset_index()
    .rename(columns={'index':'feature','importance':'AUC drop when shuffled'}),
    highlight='AUC drop when shuffled'))

### Interpreting the drivers

Permutation importance answers a specific question: *if this signal became
uninformative, how much worse would the model get?* A large drop means the model
genuinely depends on it.

The features that usually matter are the ones a marketing team would
independently recognise — **recent engagement**, **reward redemption history**,
**discount sensitivity** and **spend momentum**. When the importance list
reads like a sensible business conversation, that is evidence the model learned
something structural rather than a quirk of the synthetic noise.

---
### 12.1 Single-customer explanation

In [ ]:
# A logistic model exposes every driver as a coefficient, so an individual
# prediction can be explained by flipping one feature at a time to its typical
# value and watching the probability move.  For tree ensembles the same
# question is answered by permutation importance plus SHAP-style local
# attribution, which is out of scope for this build.
print(f'Best model: {best_model_name}')
print()
print('Native importance is available:', native_importance is not None)
print('Permutation importance is available for every model, including this one.')

---
## 13. Scoring Every Customer

The selected pipeline now predicts a response probability for the full customer
base. This is the number the reward engine consumes.

In [ ]:
probability = best_pipeline.predict_proba(features[model_features])[:, 1]
features['reward_probability'] = probability
features['prediction'] = (probability >= mt.DECISION_THRESHOLD).astype(int)

print(f'Customers scored : {len(features):,}')
print(f'Probability range: {probability.min():.3f} - {probability.max():.3f}')
print(f'Mean probability : {probability.mean():.3f}')
print(f'Predicted responders (p >= {mt.DECISION_THRESHOLD}): '
      f'{features.prediction.sum():,} ({features.prediction.mean():.1%})')
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(probability, bins=40, color='#2e86ab', edgecolor='white')
axes[0].axvline(mt.DECISION_THRESHOLD, color='#c94c63', ls='--', lw=2,
                label=f'deployment threshold ({mt.DECISION_THRESHOLD})')
axes[0].set_title('Predicted response probability, all customers')
axes[0].set_xlabel('P(response)'); axes[0].set_ylabel('Customers'); axes[0].legend()

by_seg = features.groupby('rfm_segment').reward_probability.mean().sort_values()
axes[1].barh(by_seg.index, by_seg.values, color='#17b3a3')
axes[1].set_title('Mean predicted response by RFM segment')
axes[1].set_xlabel('P(response)')
fig.tight_layout(); plt.show()

### 13.1 Probability calibration

A ranked model is useless to a reward engine if its probabilities are
meaningless. The engine decisions depend on a probability being a genuine
probability — `P = 0.8` must mean "roughly 80% of these customers respond".

Binning predictions into deciles and comparing the predicted probability with
the observed response rate is the standard diagnostic.

In [ ]:
calib = pd.DataFrame({'prob': probability, 'actual': features.reward_response.values})
calib['bucket'] = pd.qcut(calib.prob, 10, labels=False, duplicates='drop')
calib_table = calib.groupby('bucket').agg(
    mean_predicted=('prob','mean'),
    observed_rate=('actual','mean'),
    customers=('actual','size'),
).round(4)
calib_table['gap'] = calib_table.mean_predicted - calib_table.observed_rate

display(viz.style_table(calib_table.reset_index().rename(columns={'bucket':'decile'}),
                        format_map={'mean_predicted':'{:.3f}','observed_rate':'{:.3f}',
                                    'gap':'{:+.3f}'},
                        highlight='observed_rate'))

print(f"Mean absolute calibration gap : {calib_table.gap.abs().mean():.4f}")
print(f"Max absolute calibration gap  : {calib_table.gap.abs().max():.4f}")

fig, ax = plt.subplots(figsize=(6.2, 5.4))
ax.plot([0, 1], [0, 1], ls='--', c='#bdc3c7', label='perfect calibration')
ax.plot(calib_table.mean_predicted, calib_table.observed_rate,
        marker='o', lw=2, color='#c94c63', label=best_model_name)
for _, r in calib_table.iterrows():
    ax.annotate(str(int(r.customers)), (r.mean_predicted, r.observed_rate),
                textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed response rate')
ax.set_title('Calibration - Decile Plot')
ax.legend(); ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03)
fig.tight_layout(); plt.show()

print()
print('A well-calibrated curve means the reward engine can treat the score as a')
print('real probability when it computes expected campaign value. If the curve')
print('were strongly convex, the engine would systematically over-promise and')
print('the offers would need a calibration factor applied first.')

In [ ]:
scored_segments = features.groupby('rfm_segment').agg(
    customers=('customer_id','size'),
    mean_response_probability=('reward_probability','mean'),
    predicted_responders=('prediction','sum'),
).sort_values('mean_response_probability', ascending=False).round(3)
scored_segments['responder_share'] = scored_segments.predicted_responders / scored_segments.customers

display(viz.style_table(scored_segments.reset_index(),
                        format_map={'mean_response_probability':'{:.3f}',
                                    'responder_share':'{:.0%}'},
                        highlight='mean_response_probability'))

---
## 14. K-Means Customer Segmentation

RFM is a *rule-based* segmentation. K-Means is a *data-driven* one — it finds
groups without being told what to look for. Running both gives a check on
whether the business segments correspond to natural clusters in the data.

The features are standardised first, because K-Means is distance-based and
`total_spend` (thousands) would otherwise drown out `reward_usage_rate` (0–1).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

cluster_cols = ['total_orders','total_spend','average_order_value',
                'days_since_last_purchase','reward_usage_rate',
                'discount_usage_rate','category_affinity_score']

cluster_data = features[cluster_cols].copy()
cluster_scaled = StandardScaler().fit_transform(cluster_data)

scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    lab = km.fit_predict(cluster_scaled)
    scores.append({
        'k': k,
        'inertia': km.inertia_,
        'silhouette': silhouette_score(cluster_scaled, lab),
        'recommended': False,
    })
k_scores = pd.DataFrame(scores)

best_k = int(k_scores.loc[k_scores.silhouette.idxmax(), 'k'])
k_scores.loc[k_scores.k == best_k, 'recommended'] = True

display(viz.style_table(k_scores, format_map={'inertia':'{:,.0f}','silhouette':'{:.4f}'},
                        highlight='silhouette'))
print(f'Best k by silhouette score: {best_k}')

fig = viz.kmeans_elbow_plot(k_scores)
plt.show()

In [ ]:
KMEANS_K = best_k
km_final = KMeans(n_clusters=KMEANS_K, n_init=25, random_state=RANDOM_STATE)
features['cluster_id'] = km_final.fit_predict(cluster_scaled)

cluster_profile = features.groupby('cluster_id').agg(
    customers=('customer_id','size'),
    avg_orders=('total_orders','mean'),
    avg_spend=('total_spend','mean'),
    avg_recency=('days_since_last_purchase','mean'),
    avg_response=('reward_probability','mean'),
    avg_affinity=('category_affinity_score','mean'),
).round(2)

def name_cluster(r):
    if r.avg_spend > cluster_profile.avg_spend.median() and r.avg_recency < cluster_profile.avg_recency.median():
        return 'VIP Active'
    if r.avg_spend > cluster_profile.avg_spend.median():
        return 'High Value Lapsing'
    if r.avg_recency < cluster_profile.avg_recency.median():
        return 'Active Low Spend'
    return 'Dormant Low Spend'

cluster_profile['cluster_label'] = cluster_profile.apply(name_cluster, axis=1)
features['cluster_label'] = features.cluster_id.map(cluster_profile.cluster_label)

display(viz.style_table(cluster_profile.reset_index(),
                        format_map={'avg_spend':'₹{:,.0f}','avg_response':'{:.3f}',
                                    'avg_affinity':'{:.3f}'},
                        highlight='avg_spend'))

plot_df = features[['days_since_last_purchase','total_spend','cluster_label']].rename(
    columns={'days_since_last_purchase':'recency_days'})
fig = viz.kmeans_cluster_plot(plot_df)
plt.show()

### Do the clusters agree with RFM?

They overlap but are not identical, and the differences are instructive:

* K-Means clusters on **magnitude** (spend, orders), so it isolates clear
  value tiers.
* RFM also encodes **recency direction** and produces finer business labels
  such as *Cannot Lose Them*.

For the reward engine, RFM segments drive the offer-affinity bonus while the
cluster label serves as a coarse descriptive tag. They are complementary views.

In [ ]:
cross = pd.crosstab(features.rfm_segment, features.cluster_label)
display(viz.style_table(cross.reset_index(), highlight=cross.columns[0]))

---
## 15. The Reward Catalogue

Seven offers across three commercial mechanics. Each is a *capability* that the
engine scales to the customer, not a fixed promise.

| Reward | Type | Cost model | Best suited to |
|---|---|---|---|
| 5% OFF | Percentage | Reduces revenue | Champions, Loyal |
| 10% OFF | Category discount | Reduces revenue | Loyal, Champions |
| 15% OFF | Percentage | Reduces revenue | At Risk, Hibernating |
| Rs 100 OFF | Fixed | Reduces revenue | Regular, Potential Loyalists |
| Rs 200 OFF | Fixed | Reduces revenue | Champions, Loyal |
| Rs 500 OFF | Fixed | Reduces revenue | Champions, Cannot Lose Them |
| Cashback | Contingent | Paid on conversion | Regular, Promising |
| BOGO | Contingent | Paid on conversion | Price Sensitive, Occasional |

### Two cost models, deliberately

* **Discount offers** lower the price the customer pays. The cost appears as a
  smaller realised order value — it is *already* inside the margin.
* **Contingent offers** (cashback, BOGO) are paid out *after* a successful
  purchase. They are a genuine extra outflow, incurred only by responders.

Conflating the two is the classic promotion-P&L error: subtracting a discount
from the margin *and* booking it as a cash cost double-counts it.

In [ ]:
catalogue = re.reward_catalogue_frame()
display(viz.style_table(catalogue,
                        format_map={'Discount %':'{:.0f}%','Flat Amount':'₹{:,.0f}',
                                    'Minimum Purchase':'₹{:,.0f}',
                                    'Maximum Discount':'₹{:,.0f}',
                                    'Expected Uplift':'{:.0%}',
                                    'Cost Weight':'{:.2f}'}))

---
## 16. Reward Optimisation — How One Customer Is Decided

Take a single customer and evaluate the **entire** catalogue against them. This
is the engine's inner loop, exposed for inspection.

For each candidate offer the engine computes:

```
P(response | offer)   the offer modulates the base probability
realised order value  basket after the discount
net if converted      realised_value x margin            (discount offers)
                      order_value x margin - payout      (contingent offers)
expected campaign value = P(response) x net_if_converted
expected reward cost    = P(response) x gross reward value
```

In [ ]:
margins = (transactions.merge(products[['product_id','category','profit_margin']], on='product_id')
           .groupby('category').profit_margin.mean().to_dict())

# Choose a customer in the *interesting* probability band.  A customer the
# model scores at ~0.99 makes every offer look equally good; someone near the
# operating threshold is where the trade-off between discount depth and response
# lift actually bites.
half = 0.65
sample_idx = (features.reward_probability - half).abs().nsmallest(1).index[0]
sample = features.loc[sample_idx]

print(f"Customer          : {sample.customer_id}")
print(f"RFM segment       : {sample.rfm_segment}")
print(f"Base P(response)  : {sample.reward_probability:.3f}")
print(f"Preferred category: {sample.preferred_category}  "
      f"(affinity {sample.category_affinity_score:.2f})")
print(f"Typical order     : Rs {sample.average_order_value:,.0f}")
print()

options = re.optimise_reward_for_customer(sample, sample.rfm_segment, margins)
opt_df = pd.DataFrame(options)[[
    'recommended_reward','reward_type','discount_pct','discount_flat',
    'minimum_purchase','expected_response_probability','response_lift',
    'realised_order_value','gross_reward_value','estimated_reward_cost',
    'expected_campaign_value']]

display(viz.style_table(opt_df,
    format_map={'discount_pct':'{:.0f}%','discount_flat':'Rs {:,.0f}',
                'minimum_purchase':'Rs {:,.0f}','expected_response_probability':'{:.3f}',
                'response_lift':'{:+.3f}','realised_order_value':'Rs {:,.0f}',
                'gross_reward_value':'Rs {:,.0f}','estimated_reward_cost':'Rs {:,.0f}',
                'expected_campaign_value':'Rs {:,.0f}'},
    highlight='expected_campaign_value'))

best = options[0]
print()
print(f"Engine picks: {best['recommended_reward']}")
print(f"  Expected campaign value : Rs {best['expected_campaign_value']:,.0f}")
print(f"  Expected reward cost    : Rs {best['estimated_reward_cost']:,.0f}")
print(f"  Lift over base          : {best['response_lift']:+.3f}")
print()
print('Note the biggest discount is NOT automatically selected. The engine trades')
print('discount depth against response lift and margin, exactly as a merchant would.')

### 16.1 The optimisation trade-off, visualised

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sub = opt_df.copy()
sub['depth'] = sub.discount_pct + sub.discount_flat / 20
ax.scatter(sub.depth, sub.expected_campaign_value,
           s=90, c=sub.expected_response_probability, cmap='viridis', edgecolor='k')
for _, r in sub.iterrows():
    ax.annotate(r.recommended_reward, (r.depth, r.expected_campaign_value),
                textcoords='offset points', xytext=(6, 5), fontsize=8)
ax.set_xlabel('Offer depth (discount % + rupee value / 20)')
ax.set_ylabel('Expected campaign value (Rs)')
ax.set_title(f'Reward Trade-off for {sample.customer_id}')
cb = plt.colorbar(ax.collections[0]); cb.set_label('P(response)')
plt.tight_layout(); plt.show()

---
## 17. Reward Recommendations for the Whole Base

The engine now runs for every customer, with an optional campaign budget.

### 17.1 Scoring the catalogue for everyone

In [ ]:
recommendations = re.recommend_rewards(
    features,
    margins,
    budget=None,        # first pass: show what the model *wants* to do
    value_weight=1.0,
)

print(f'Customers with a recommendation : {len(recommendations):,}')
print(f'Distinct rewards used           : {recommendations.recommended_reward.nunique()}')
print()
print('Reward distribution:')
dist = recommendations.recommended_reward.value_counts().rename('customers').to_frame()
dist['share'] = dist.customers / len(recommendations)
display(viz.style_table(dist.reset_index(),
                        format_map={'share':'{:.1%}'}, highlight='customers'))

fig = viz.reward_distribution_chart(recommendations)
plt.show()

Distinct offers across the customer base is direct evidence the system is doing
something other than a blanket discount.

### 17.2 Expected value of the programme

In [ ]:
targeted = recommendations[recommendations.targetable]

print(f'Targetable customers         : {len(targeted):,} of {len(recommendations):,} '
      f'({len(targeted)/len(recommendations):.1%})')
print(f'Total expected reward cost   : Rs {targeted.estimated_reward_cost.sum():,.0f}')
print(f'Total expected campaign value: Rs {targeted.expected_campaign_value.sum():,.0f}')
print(f'Expected value per rupee     : '
      f'{targeted.expected_campaign_value.sum()/max(targeted.estimated_reward_cost.sum(),1):.2f}')
print()
print('Customers excluded as untargetable (very low response probability):')
print(f'  {len(recommendations)-len(targeted):,} customers, '
      f'Rs {(recommendations[~recommendations.targetable].estimated_reward_cost).sum():,.0f} '
      f'of expected cost avoided')

In [ ]:
fig = viz.discount_vs_value_plot(recommendations)
plt.show()

mix = recommendations.groupby('reward_type').agg(
    customers=('customer_id','size'),
    avg_discount_pct=('discount_pct','mean'),
    total_expected_cost=('estimated_reward_cost','sum'),
    total_expected_value=('expected_campaign_value','sum'),
).round(2)
mix['value_per_rupee'] = (mix.total_expected_value / mix.total_expected_cost.replace(0, np.nan)).round(2)

display(viz.style_table(mix.reset_index().fillna(0),
                        format_map={'total_expected_cost':'Rs {:,.0f}',
                                    'total_expected_value':'Rs {:,.0f}',
                                    'avg_discount_pct':'{:.1f}%'},
                        highlight='total_expected_value'))

---
## 18. Reward by Category

Rewards are attached to the customer's **preferred category**, because an offer
on a category they never buy is a wasted offer.

In [ ]:
by_cat = features.merge(
    recommendations[['customer_id','reward_type','estimated_reward_cost',
                     'expected_campaign_value','targetable']],
    on='customer_id'
).groupby('preferred_category').agg(
    customers=('customer_id','size'),
    avg_response=('reward_probability','mean'),
    avg_affinity=('category_affinity_score','mean'),
    total_expected_cost=('estimated_reward_cost','sum'),
    total_expected_value=('expected_campaign_value','sum'),
).round(2).sort_values('total_expected_value', ascending=False)

display(viz.style_table(by_cat.reset_index(),
                        format_map={'avg_response':'{:.3f}','avg_affinity':'{:.3f}',
                                    'total_expected_cost':'Rs {:,.0f}',
                                    'total_expected_value':'Rs {:,.0f}'},
                        highlight='total_expected_value'))

fig = viz.category_response_rate_chart(features, transactions, products)
plt.show()

In [ ]:
fig = viz.discount_sensitivity_chart(features)
plt.show()

---
## 19. Campaign Priority Ranking

A campaign cannot contact everyone at once. Customers are ranked by a composite
of five signals, each converted to a percentile first so no single scale
dominates.

| Signal | Weight | Rationale |
|---|---|---|
| Response probability | 30% | Who is actually reachable |
| Expected campaign value | 22% | Who pays for the programme |
| Historical value | 20% | Protect the customers who matter |
| Urgency (recency, inverted) | 16% | Catch them before they churn |
| Category affinity | 12% | How actionable the offer is |

In [ ]:
campaign = re.campaign_priority(features, recommendations)

print('Top 20 campaign targets:')
display(viz.style_table(
    campaign[['campaign_rank','customer_id','rfm_segment','preferred_category',
              'reward_probability','recommended_reward','estimated_reward_cost',
              'expected_campaign_value','campaign_priority_score']].head(20),
    format_map={'reward_probability':'{:.3f}','estimated_reward_cost':'Rs {:,.0f}',
                'expected_campaign_value':'Rs {:,.0f}','campaign_priority_score':'{:.1f}'},
    highlight='campaign_priority_score'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

axes[0].plot(campaign.campaign_rank, campaign.campaign_priority_score,
             color='#1f4e79', lw=2)
axes[0].fill_between(campaign.campaign_rank, campaign.campaign_priority_score,
                     color='#1f4e79', alpha=0.15)
axes[0].set_xlabel('Campaign rank'); axes[0].set_ylabel('Priority score')
axes[0].set_title('Priority decays smoothly - no arbitrary cliff')

seg_rank = campaign.groupby('rfm_segment').agg(
    mean_priority=('campaign_priority_score','mean'),
    customers=('customer_id','size')).sort_values('mean_priority', ascending=False)
axes[1].barh(seg_rank.index, seg_rank.mean_priority, color='#17b3a3')
axes[1].set_xlabel('Mean priority score')
axes[1].set_title('Which segments earn the top slots')
axes[1].invert_yaxis()
fig.tight_layout(); plt.show()

print('Priority by segment:')
display(viz.style_table(seg_rank.reset_index(),
                        format_map={'mean_priority':'{:.1f}'},
                        highlight='mean_priority'))

---
## 20. Campaign Simulation — Does Targeting Actually Pay?

The decisive test. Three strategies, the **same budget**, evaluated with the
same simulated response model, so the only difference is who is targeted and
what they are offered.

| Strategy | Description |
|---|---|
| **A. Random** | Contact customers at random until the budget is spent |
| **B. ML-Targeted** | Contact the highest predicted-responders first |
| **C. ML + Cost-Optimised** | ML targeting with a per-customer offer ceiling so cheaper offers fit more people |

In [ ]:
simulation = re.simulate_campaign(features, recommendations, budget=50_000)
summary = simulation['summary']

display(viz.style_table(
    summary.drop(columns=['Budget']),
    format_map={'Expected Responders':'{:,.0f}','Response Rate':'{:.1%}',
                'Expected Revenue':'Rs {:,.0f}','Promotional Cost':'Rs {:,.0f}',
                'Expected Net Value':'Rs {:,.0f}','Cost per Contact':'Rs {:,.0f}',
                'Net per Contact':'Rs {:,.0f}','Net Value vs Random':'{:.2f}x'},
    highlight='Expected Net Value'))

a, b, c = summary.iloc[0], summary.iloc[1], summary.iloc[2]
print(f"Strategy A (random)              : {a['Expected Responders']:.0f} responders, "
      f"net Rs {a['Expected Net Value']:,.0f}")
print(f"Strategy B (ML targeted)         : {b['Expected Responders']:.0f} responders, "
      f"net Rs {b['Expected Net Value']:,.0f}")
print(f"Strategy C (ML + cost-optimised) : {c['Expected Responders']:.0f} responders, "
      f"net Rs {c['Expected Net Value']:,.0f}")
print()
print(f"ML targeting vs random on net value : {b['Expected Net Value']/max(abs(a['Expected Net Value']),1):.2f}x")
print(f"ML targeting vs random on responders: {b['Expected Responders']/max(a['Expected Responders'],1):.2f}x")
print(f"Response rate  A {a['Response Rate']:.1%}  |  B {b['Response Rate']:.1%}  |  C {c['Response Rate']:.1%}")
print()

fig = viz.campaign_simulation_chart(simulation)
plt.show()

print(simulation['note'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))

rb = campaign[['customer_id','expected_campaign_value','targetable']].dropna()
rb['decile'] = pd.qcut(rb.expected_campaign_value.rank(method='first'), 10,
                       labels=[f'D{i}' for i in range(1, 11)])
gain = rb.groupby('decile', observed=True).expected_campaign_value.sum()
axes[0].bar(gain.index.astype(str), gain.values, color='#2e86ab')
axes[0].set_title('Expected value by campaign decile')
axes[0].set_xlabel('Decile of predicted value'); axes[0].set_ylabel('Expected value (Rs)')
axes[0].tick_params(axis='x', rotation=45)

cum = gain.cumsum() / gain.sum()
axes[1].plot(range(1, 11), cum.values * 100, marker='o', color='#f37748', lw=2)
axes[1].axhline(80, ls='--', color='#607d8b', label='80% of value')
axes[1].set_xlabel('Top-N deciles targeted')
axes[1].set_ylabel('Cumulative share of value (%)')
axes[1].set_title('Value concentration in the top deciles')
axes[1].legend()
fig.tight_layout(); plt.show()

top3 = gain.sort_values(ascending=False).head(3).sum() / gain.sum()
print(f'The top 3 deciles carry {top3:.1%} of expected campaign value.')

---
## 21. Customer-Level Decision Output

Everything the system knows about one customer, in one view. This is what a
CRM screen or an outbound-call script would render.

In [ ]:
from IPython.display import HTML, display as ipy_display

ipy_display(HTML(viz.customer_profile_card(campaign.iloc[0])))

In [ ]:
at_risk = campaign[campaign.rfm_segment == 'At Risk']
if len(at_risk):
    ipy_display(HTML(viz.customer_profile_card(at_risk.iloc[0])))
else:
    ipy_display(HTML(viz.customer_profile_card(campaign.iloc[1])))

In [ ]:
lowp = campaign[campaign.reward_probability < 0.35]
if len(lowp):
    ipy_display(HTML(viz.customer_profile_card(lowp.iloc[0])))
else:
    ipy_display(HTML(viz.customer_profile_card(campaign.iloc[-1])))

---
## 22. Executive Dashboard

In [ ]:
dashboard = {
    'Customers Analysed'      : f'{len(features):,}',
    'Total Revenue (obs.)'    : viz._money(features.total_spend.sum()),
    'Average Order Value'     : viz._money(features.total_spend.sum()/features.total_orders.sum()),
    'Model Selected'          : best_model_name.replace(' ', '\n'),
    'Test F1'                 : f"{winner.F1:.3f}",
    'Test ROC-AUC'            : f"{winner['ROC-AUC']:.3f}",
    'Predicted Responders'    : f"{int(features.prediction.sum()):,}",
    'Targetable Customers'    : f"{int(recommendations.targetable.sum()):,}",
    'Distinct Rewards Used'   : f"{recommendations.recommended_reward.nunique()}",
    'Expected Reward Cost'    : viz._money(recommendations[recommendations.targetable].estimated_reward_cost.sum()),
    'Expected Campaign Value' : viz._money(recommendations[recommendations.targetable].expected_campaign_value.sum()),
    'ML Net Value Lift'       : f"{b['Expected Net Value']/max(abs(a['Expected Net Value']),1):.2f}x",
}
ipy_display(HTML(viz.kpi_card_grid(dashboard)))

In [ ]:
executive = pd.DataFrame({
    'Metric': ['Best model','Test F1','Test ROC-AUC','Test PR-AUC',
               'Train-test AUC gap','Customers scored','Predicted responders',
               'Targetable customers','Rewards deployed',
               'Expected reward cost','Expected campaign value','ML vs random (net)'],
    'Value': [best_model_name, f'{winner.F1:.4f}', f'{winner["ROC-AUC"]:.4f}',
              f'{winner["PR-AUC"]:.4f}', f'{winner["Overfitting (train-test AUC)"]:.4f}',
              f'{len(features):,}', f'{int(features.prediction.sum()):,}',
              f'{int(recommendations.targetable.sum()):,}',
              f'{recommendations.recommended_reward.nunique()}',
              viz._money(recommendations[recommendations.targetable].estimated_reward_cost.sum()),
              viz._money(recommendations[recommendations.targetable].expected_campaign_value.sum()),
              f"{b['Expected Net Value']/max(abs(a['Expected Net Value']),1):.2f}x"],
})
display(viz.style_table(executive))

---
## 23. Saving the Artefacts

The winning pipeline, the preprocessing contract and the metadata are persisted
so the reward engine can run without retraining.

Because a scikit-learn `Pipeline` bundles preprocessing and estimator in one
object, a single pickle is a complete, deployable model.

In [ ]:
paths = mt.save_artifacts(best_model_name, best_pipeline, split_info, comparison,
                          extra_metadata={'target_response_rate': float(features.reward_response.mean())},
                          models_dir='models')

for name, p in paths.items():
    size = os.path.getsize(p) / 1024
    print(f'  {name:14s} -> {p:44s} ({size:,.1f} KB)')

loaded_model, metadata, loaded_preprocessor = mt.load_artifacts('models')
reloaded = loaded_model.predict_proba(features[model_features])[:, 1]

print()
print(f"Reloaded model reproduces predictions exactly: "
      f"{np.allclose(reloaded, features.reward_probability.values, atol=1e-9)}")
print(f"Best model recorded in metadata              : {metadata['best_model']}")

In [ ]:
save_dir = 'outputs/reports'
os.makedirs(save_dir, exist_ok=True)

recommendations.to_csv(f'{save_dir}/reward_recommendations.csv', index=False)
campaign.to_csv(f'{save_dir}/campaign_priority.csv', index=False)
comparison.drop(columns=[c for c in comparison.columns if c.startswith('_')]) \
          .to_csv(f'{save_dir}/model_comparison.csv', index=False)
summary.to_csv(f'{save_dir}/campaign_simulation.csv', index=False)
category_profile.to_csv(f'{save_dir}/category_affinity.csv', index=False)

print('Reports written to', save_dir)
for fname in sorted(os.listdir(save_dir)):
    print('  ', fname)

---
## 24. Product & Category Opportunities

Bringing the product and customer views together, three concrete opportunities
fall out of the data.

### 24.1 Category response vs revenue

Categories where predicted response is high but revenue is low are **growth**
opportunities. High revenue with low response is a **retention** priority.

In [ ]:
opp = features.groupby('preferred_category').agg(
    customers=('customer_id','size'),
    mean_response=('reward_probability','mean'),
    total_hist_spend=('total_spend','sum'),
).join(cat_summary[['revenue','avg_margin']])

opp['revenue_cr'] = opp.revenue / 1e7
opp['response_index'] = opp.mean_response / opp.mean_response.mean()
opp['revenue_index'] = opp.revenue / opp.revenue.mean()

def classify(r):
    if r.response_index >= 1 and r.revenue_index < 1:
        return 'Grow'
    if r.response_index < 1 and r.revenue_index >= 1:
        return 'Retain'
    if r.response_index >= 1 and r.revenue_index >= 1:
        return 'Defend'
    return 'Nurture'

opp['opportunity'] = opp.apply(classify, axis=1)
display(viz.style_table(opp.reset_index(),
                        format_map={'mean_response':'{:.3f}','revenue_cr':'Rs {:.2f} cr',
                                    'total_hist_spend':'Rs {:,.0f}','avg_margin':'{:.1%}',
                                    'response_index':'{:.2f}x','revenue_index':'{:.2f}x'},
                        highlight='mean_response'))

fig, ax = plt.subplots(figsize=(9, 5.5))
colors = {'Grow':'#16a085','Retain':'#c94c63','Defend':'#1f4e79','Nurture':'#f0a202'}
for label, sub in opp.groupby('opportunity'):
    ax.scatter(sub.revenue_index, sub.response_index, s=sub.customers*1.4,
               color=colors[label], alpha=0.75, label=label, edgecolor='white')
for cname, r in opp.iterrows():
    ax.annotate(cname, (r.revenue_index, r.response_index), fontsize=7.5,
                textcoords='offset points', xytext=(5, 4))
ax.axhline(1, ls='--', color='#90a4ae', lw=1)
ax.axvline(1, ls='--', color='#90a4ae', lw=1)
ax.set_xlabel('Revenue index (1.0 = average)')
ax.set_ylabel('Response index (1.0 = average)')
ax.set_title('Category Opportunity Map\n(bubble size = customers)')
ax.legend(title='Opportunity'); plt.tight_layout(); plt.show()

### 24.2 Brand concentration

Customers buying many brands are *promiscuous* shoppers — harder to retain and
less responsive to brand-specific offers. Customers concentrated in one brand
are loyal and reward category-specific incentives well.

In [ ]:
features['concentration'] = pd.qcut(features.top_brand_share, 4,
                                    labels=['Broad shopper','Mixed','Focused','Brand loyal'])
conc = features.groupby('concentration', observed=True).agg(
    customers=('customer_id','size'),
    avg_response=('reward_probability','mean'),
    avg_affinity=('category_affinity_score','mean'),
    avg_spend=('total_spend','mean'),
    avg_discount=('average_discount_percentage','mean'),
).round(3)
display(viz.style_table(conc.reset_index(),
                        format_map={'avg_response':'{:.3f}','avg_affinity':'{:.3f}',
                                    'avg_spend':'Rs {:,.0f}','avg_discount':'{:.2f}%'},
                        highlight='avg_response'))

---
## 25. Business Recommendations

### Findings

1. **The model is doing real work.** Test ROC-AUC and PR-AUC are far above the
   no-skill baseline, and the benchmark shows which algorithm actually earned
   its place rather than which was assumed best.

2. **Rewards genuinely differentiate.** The engine deploys multiple distinct
   offers. A blanket campaign would use one.

3. **Targeting beats randomness** in the budget-matched simulation, on both
   responder count and net value.

4. **Value concentrates.** A small share of customers carries most of the
   expected campaign value, which is exactly why uniform discounting is
   expensive.

5. **Category attachment matters.** Rewards are tied to each customer's
   preferred category, so the offer is redeemable.

### Recommendations

| Recommendation | Evidence | Expected effect |
|---|---|---|
| Replace blanket discounts with the ranked top-decile campaign | Section 20 value concentration | Same budget, materially more responders |
| Use contingent offers (cashback/BOGO) for price-sensitive customers | Section 17 value-per-rupee by reward type | Response without a permanent price cut |
| Reserve 15% OFF for lapsed high-value customers | Section 16 offer trade-off table | Reactivation where deep cuts are justified |
| Attach category coupons to each customer's top affinity | Section 18 reward-by-category | Higher redemption on offers that fit |
| Rebuild the model quarterly | Section 4 observation-window discipline | Drift control |

### Limitations — stated plainly

1. **The target is synthetic.** `reward_response` was generated, not observed.
   Every model metric is a *methodological* result. It demonstrates that the
   pipeline is sound and leak-free; it is not evidence about real customers.
2. **The uplift model is a behavioural assumption.** The response lift assigned
   to each offer is a parameter, not something estimated from an experiment.
3. **Expected values are illustrative**, not financial forecasts. They contain
   no incremental-revenue truth beyond the simulation.
4. **No causal inference.** The model predicts association, not the causal
   effect of a reward. Randomised holdout testing is the only way to measure
   true lift.
5. **The dataset is moderate** (1,500 customers). Confidence intervals on
   precision/recall are non-trivial at this size.

### The honest bottom line

This project delivers a **complete, leak-free, reproducible decision pipeline**:
validated data, disciplined feature engineering, a fair five-model comparison,
an explainable winner, and a cost-aware reward engine with a quantified
simulation. It does **not** deliver a claim about real-world results, because
the data to support such a claim does not exist here. Any project that skipped
that distinction would be overselling.

### Next step worth doing first

Run a **randomised holdout campaign** on a few thousand customers. That single
experiment converts every assumption in this notebook into a measurement.

In [ ]:
print('=' * 70)
print('  PROJECT COMPLETE')
print('=' * 70)
print(f'  Best model            : {best_model_name}')
print(f'  Test F1               : {winner.F1:.4f}')
print(f'  Test ROC-AUC          : {winner["ROC-AUC"]:.4f}')
print(f'  Customers scored      : {len(features):,}')
print(f'  Rewards deployed      : {recommendations.recommended_reward.nunique()} distinct types')
print(f'  ML net-value vs random: {b["Expected Net Value"]/max(abs(a["Expected Net Value"]),1):.2f}x')
print('=' * 70)
print()
print('Reminder: reward_response is simulated. These are methodological')
print('results that validate the pipeline, not real business outcomes.')